# Offenbach energy data: first inspection

No model or data-quality claims yet. Read `docs/data-and-evaluation.md` and the dataset's README and issue logs before choosing a target. This notebook does not download data.

Set `ENERGY_DATA_CSV` to one extracted **hourly** CSV or gzip CSV. Dataset files remain outside Git.

In [ ]:
import os
from pathlib import Path
import pandas as pd

selected = os.environ.get('ENERGY_DATA_CSV')
if not selected:
    raise RuntimeError('Set ENERGY_DATA_CSV to an extracted hourly CSV, then restart the kernel.')
path = Path(selected).expanduser()
if not path.is_file():
    raise FileNotFoundError(path)
frame = pd.read_csv(path, compression='infer')
print(f'{len(frame):,} rows; {len(frame.columns)} columns')
display(frame.head())
display(frame.dtypes.to_frame('dtype'))


## Schema and missingness

Verify delimiter, timestamp column, units and interval conventions against the source README. A single unexpected column may indicate the wrong delimiter. Missing values shown here do not identify every corrected reading; consult issue logs.

In [ ]:
display(frame.isna().mean().mul(100).sort_values(ascending=False).to_frame('missing_percent'))
display(frame.describe(include='all').T)


## Record before continuing

- Source version, file checksum, meter name and measurement unit.
- Is this total demand, grid exchange, generation or a cumulative meter?
- Timestamp timezone and interval-start/end meaning.
- Measured versus inferred/corrected intervals and usable coverage.
- PV commissioning, shutdowns and other regime changes.

Only after these are understood: normalise hourly intervals, plot coverage and weekly profiles, select a chronological split and evaluate the seasonal-naive baselines. Do not use future observations as features.